In [1]:
# =================================================================================================
# NHANES 2013-2014 DIETARY MERGE - FIDELITY PRESERVING
#
# Source files:
#
#   DR1TOT_H_corrected.csv
#   DR2TOT_H_corrected.csv
#
# Purpose:
#
#   Merge NHANES 2013-2014 Dietary Day 1 and Day 2
#   Total Nutrient Intake files into one participant-level CSV.
#
#
# FIDELITY PRINCIPLES
# -------------------------------------------------------------------------------------------------
#
#   1. Read corrected CSVs as TEXT.
#
#   2. Preserve Dietary decimal representation AS IS.
#
#   3. Do NOT round.
#
#   4. Do NOT use float_format=.
#
#   5. Do NOT integerize nutrient measurements.
#
#   6. Validate Day 1 / Day 2 shared variables before deduplicating.
#
#   7. Retain ONE canonical copy of:
#
#          WTDRD1
#          WTDR2D
#          DRABF
#          DRDINT
#
#   8. Prevent engineered duplicate fields such as:
#
#          WTDRD1_DAY_2_TOTAL_INTAKE
#          WTDR2D_DAY_2_TOTAL_INTAKE
#          DRABF_DAY_2_TOTAL_INTAKE
#          DRDINT_DAY_2_TOTAL_INTAKE
#
#   9. Preserve legitimate numeric zeros.
#
#  10. Validate exact source-to-merged text fidelity.
#
#  11. Validate exact pre-save/post-save CSV text fidelity.
#
#
# CDC 2013-2014 BENCHMARKS
# -------------------------------------------------------------------------------------------------
#
#   Total rows in DR1TOT_H / DR2TOT_H       = 9,813
#
#   WTDRD1:
#       positive Day 1 weight               = 8,661
#       zero (Day 1 incomplete/not done)    = 1,152
#       missing                             =     0
#
#   WTDR2D:
#       positive two-day weight             = 7,574
#       zero (Day 2 incomplete/not done)    = 1,087
#       missing                             = 1,152
#
#   DRDINT:
#       1 = Day 1 only                      = 1,087
#       2 = Day 1 + Day 2                   = 7,574
#       missing                             = 1,152
#
#   DRABF:
#       1 = Yes                             =   134
#       2 = No                              = 8,527
#       missing                             = 1,152
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Dietary"
)


DAY1_FILE = (
    DATA_DIR
    / "DR1TOT_H_corrected.csv"
)


DAY2_FILE = (
    DATA_DIR
    / "DR2TOT_H_corrected.csv"
)


OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed_v3"
    / "2013_2014"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTPUT_PATH = (
    OUTPUT_DIR
    / "neighbourhood_nhanes_dietary_2013_2014_raw_corrected.csv"
)


# =================================================================================================
# 3. READ CORRECTED FILES AS TEXT
# =================================================================================================

#
# CRITICAL:
#
# dtype=str protects already-corrected Dietary decimal tokens.
#
# Example:
#
#     0.028999999999
#
# remains:
#
#     0.028999999999
#
# We are merging.
# We are NOT recalculating Dietary measurements.
#

day1 = pd.read_csv(
    DAY1_FILE,
    dtype=str,
    keep_default_na=False
)


day2 = pd.read_csv(
    DAY2_FILE,
    dtype=str,
    keep_default_na=False
)


print("=" * 110)

print(
    "NHANES 2013-2014 DIETARY DAY 1 + DAY 2 MERGE"
)

print("=" * 110)


print(
    "DR1TOT_H shape:",
    day1.shape
)


print(
    "DR2TOT_H shape:",
    day2.shape
)


# =================================================================================================
# 4. STANDARDIZE COLUMN NAMES
# =================================================================================================

day1.columns = (
    day1.columns
    .str.strip()
    .str.upper()
)


day2.columns = (
    day2.columns
    .str.strip()
    .str.upper()
)


# =================================================================================================
# 5. CYCLE-SPECIFIC CDC BENCHMARKS
# =================================================================================================

EXPECTED_ROWS = 9813


# WTDRD1
EXPECTED_WTDRD1_POSITIVE = 8661
EXPECTED_WTDRD1_ZERO = 1152
EXPECTED_WTDRD1_MISSING = 0


# WTDR2D
EXPECTED_WTDR2D_POSITIVE = 7574
EXPECTED_WTDR2D_ZERO = 1087
EXPECTED_WTDR2D_MISSING = 1152


EXPECTED_DRDINT = {

    "1": 1087,

    "2": 7574,

    "": 1152

}


EXPECTED_DRABF = {

    "1": 134,

    "2": 8527,

    "": 1152

}


# =================================================================================================
# 6. VERIFY SEQN EXISTS
# =================================================================================================

for name, dataframe in [

    ("DR1TOT_H", day1),

    ("DR2TOT_H", day2)

]:

    if "SEQN" not in dataframe.columns:

        raise ValueError(
            f"{name}: SEQN is missing."
        )


print(
    "PASS: SEQN exists in both files."
)


# =================================================================================================
# 7. VERIFY ROW COUNTS
# =================================================================================================

if len(day1) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected DR1TOT_H row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(day1):,}"
    )


if len(day2) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected DR2TOT_H row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(day2):,}"
    )


print(
    "PASS: Both DR1TOT_H and DR2TOT_H "
    "contain 9,813 rows."
)


# =================================================================================================
# 8. SEQN VALIDATION FUNCTION
# =================================================================================================

def validate_seqn(
    dataframe,
    name
):

    print(
        f"\n--- SEQN CHECK: {name} ---"
    )


    blank_count = int(
        dataframe["SEQN"]
        .eq("")
        .sum()
    )


    duplicate_count = int(
        dataframe["SEQN"]
        .duplicated()
        .sum()
    )


    dot_zero_count = int(

        (
            dataframe["SEQN"].ne("")
            &
            dataframe["SEQN"]
            .str.endswith(".0")
        )

        .sum()

    )


    invalid_format_count = int(

        (
            dataframe["SEQN"].ne("")
            &
            ~dataframe["SEQN"]
            .str.fullmatch(r"\d+")
        )

        .sum()

    )


    print(
        "Rows:",
        f"{len(dataframe):,}"
    )


    print(
        "Unique SEQN:",
        f"{dataframe['SEQN'].nunique():,}"
    )


    print(
        "Blank SEQN:",
        blank_count
    )


    print(
        "Duplicate SEQN:",
        duplicate_count
    )


    print(
        "SEQN ending '.0':",
        dot_zero_count
    )


    print(
        "Invalid SEQN:",
        invalid_format_count
    )


    if blank_count != 0:

        raise ValueError(
            f"{name}: blank SEQN found."
        )


    if duplicate_count != 0:

        raise ValueError(
            f"{name}: duplicate SEQN found."
        )


    if dot_zero_count != 0:

        raise ValueError(
            f"{name}: SEQN contains '.0'."
        )


    if invalid_format_count != 0:

        raise ValueError(
            f"{name}: invalid SEQN format."
        )


    print(
        "PASS"
    )


# =================================================================================================
# 9. VALIDATE SEQN
# =================================================================================================

validate_seqn(
    day1,
    "DR1TOT_H"
)


validate_seqn(
    day2,
    "DR2TOT_H"
)


print(
    "\nPASS: SEQN integrity validated in both files."
)


# =================================================================================================
# 10. VERIFY SAME PARTICIPANT UNIVERSE
# =================================================================================================

day1_seqn = set(
    day1["SEQN"]
)


day2_seqn = set(
    day2["SEQN"]
)


only_day1 = (
    day1_seqn
    -
    day2_seqn
)


only_day2 = (
    day2_seqn
    -
    day1_seqn
)


print(
    "\n--- PARTICIPANT SET CHECK ---"
)


print(
    "Only in DR1TOT_H:",
    len(only_day1)
)


print(
    "Only in DR2TOT_H:",
    len(only_day2)
)


if only_day1 or only_day2:

    print(
        "\nExample Day 1-only SEQN:"
    )


    print(
        sorted(
            only_day1
        )[:20]
    )


    print(
        "\nExample Day 2-only SEQN:"
    )


    print(
        sorted(
            only_day2
        )[:20]
    )


    raise ValueError(
        "DR1TOT_H and DR2TOT_H do not contain "
        "the same participant universe."
    )


print(
    "PASS: Both files contain the same "
    "9,813 participants."
)


# =================================================================================================
# 11. DETECT SAME-NAME VARIABLES
# =================================================================================================

shared_columns_detected = [

    col

    for col in day1.columns

    if (
        col in day2.columns
        and
        col != "SEQN"
    )

]


print(
    "\n--- SHARED VARIABLES DETECTED ---"
)


print(
    shared_columns_detected
)


# =================================================================================================
# 12. EXPECTED OFFICIAL SHARED VARIABLES
# =================================================================================================

SHARED_COLUMNS = [

    "WTDRD1",

    "WTDR2D",

    "DRABF",

    "DRDINT"

]


if set(
    shared_columns_detected
) != set(
    SHARED_COLUMNS
):

    raise ValueError(
        "Unexpected shared-column pattern.\n\n"
        f"Expected:\n{SHARED_COLUMNS}\n\n"
        f"Detected:\n{shared_columns_detected}"
    )


print(
    "PASS: Exactly four expected shared variables found."
)


# =================================================================================================
# 13. BUILD SHARED-VARIABLE COMPARISON
# =================================================================================================

shared_comparison = (

    day1[
        ["SEQN"] + SHARED_COLUMNS
    ]

    .merge(

        day2[
            ["SEQN"] + SHARED_COLUMNS
        ],

        on="SEQN",

        how="outer",

        suffixes=(
            "_DAY1",
            "_DAY2"
        ),

        validate="one_to_one"

    )

)


# =================================================================================================
# 14. VALIDATE SHARED VARIABLES
# =================================================================================================

#
# Compare numerical COPIES only.
#
# Source strings themselves remain untouched.
#
# This prevents a false failure from:
#
#     "0"
# vs
#     "0.0"
#

print(
    "\n--- SHARED VARIABLE FIDELITY CHECK ---"
)


for col in SHARED_COLUMNS:

    day1_raw = (
        shared_comparison[
            f"{col}_DAY1"
        ]
    )


    day2_raw = (
        shared_comparison[
            f"{col}_DAY2"
        ]
    )


    day1_missing = (
        day1_raw.eq("")
    )


    day2_missing = (
        day2_raw.eq("")
    )


    missing_mismatch_count = int(

        (
            day1_missing
            !=
            day2_missing
        )

        .sum()

    )


    both_populated = (
        ~day1_missing
        &
        ~day2_missing
    )


    day1_numeric = pd.to_numeric(

        day1_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    day2_numeric = pd.to_numeric(

        day2_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    numeric_equal = np.isclose(

        day1_numeric.to_numpy(
            dtype=float
        ),

        day2_numeric.to_numpy(
            dtype=float
        ),

        rtol=0,

        atol=0

    )


    numeric_mismatch_count = int(

        (
            ~numeric_equal
        )

        .sum()

    )


    day1_text = (

        day1_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    day2_text = (

        day2_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    text_difference_count = int(

        (
            day1_text
            !=
            day2_text
        )

        .sum()

    )


    print(
        f"\n{col}"
    )


    print(
        "  Both populated:",
        f"{int(both_populated.sum()):,}"
    )


    print(
        "  Missing-position mismatches:",
        missing_mismatch_count
    )


    print(
        "  True numeric mismatches:",
        numeric_mismatch_count
    )


    print(
        "  Text-format differences:",
        text_difference_count
    )


    if missing_mismatch_count != 0:

        raise ValueError(
            f"{col}: missing positions differ "
            "between DR1TOT_H and DR2TOT_H."
        )


    if numeric_mismatch_count != 0:

        raise ValueError(
            f"{col}: true numeric values differ "
            "between DR1TOT_H and DR2TOT_H."
        )


print(
    "\nPASS: Shared variables agree between "
    "DR1TOT_H and DR2TOT_H."
)


# =================================================================================================
# 15. WTDRD1 FIDELITY / CDC CHECK
# =================================================================================================

#
# Unlike simply counting nonblank values, distinguish:
#
#     positive
#     zero
#     missing
#
# because zero is a legitimate official value here.
#

wtdrd1_numeric = pd.to_numeric(

    day1["WTDRD1"]
    .where(
        day1["WTDRD1"] != "",
        np.nan
    ),

    errors="raise"

)


wtdrd1_positive = int(
    (
        wtdrd1_numeric > 0
    )
    .sum()
)


wtdrd1_zero = int(
    wtdrd1_numeric
    .eq(0)
    .sum()
)


wtdrd1_missing = int(
    wtdrd1_numeric
    .isna()
    .sum()
)


print(
    "\n--- CDC WTDRD1 CHECK ---"
)


print(
    "Positive weights:",
    f"{wtdrd1_positive:,}"
)


print(
    "Zero weights:",
    f"{wtdrd1_zero:,}"
)


print(
    "Missing:",
    f"{wtdrd1_missing:,}"
)


if wtdrd1_positive != EXPECTED_WTDRD1_POSITIVE:

    raise ValueError(
        "WTDRD1 positive-weight count differs from CDC."
    )


if wtdrd1_zero != EXPECTED_WTDRD1_ZERO:

    raise ValueError(
        "WTDRD1 zero count differs from CDC."
    )


if wtdrd1_missing != EXPECTED_WTDRD1_MISSING:

    raise ValueError(
        "WTDRD1 missing count differs from CDC."
    )


print(
    "PASS: WTDRD1 matches CDC."
)


# =================================================================================================
# 16. WTDR2D FIDELITY / CDC CHECK
# =================================================================================================

wtdr2d_numeric = pd.to_numeric(

    day1["WTDR2D"]
    .where(
        day1["WTDR2D"] != "",
        np.nan
    ),

    errors="raise"

)


wtdr2d_positive = int(
    (
        wtdr2d_numeric > 0
    )
    .sum()
)


wtdr2d_zero = int(
    wtdr2d_numeric
    .eq(0)
    .sum()
)


wtdr2d_missing = int(
    wtdr2d_numeric
    .isna()
    .sum()
)


print(
    "\n--- CDC WTDR2D CHECK ---"
)


print(
    "Positive weights:",
    f"{wtdr2d_positive:,}"
)


print(
    "Zero weights:",
    f"{wtdr2d_zero:,}"
)


print(
    "Missing:",
    f"{wtdr2d_missing:,}"
)


if wtdr2d_positive != EXPECTED_WTDR2D_POSITIVE:

    raise ValueError(
        "WTDR2D positive-weight count differs from CDC."
    )


if wtdr2d_zero != EXPECTED_WTDR2D_ZERO:

    raise ValueError(
        "WTDR2D zero count differs from CDC."
    )


if wtdr2d_missing != EXPECTED_WTDR2D_MISSING:

    raise ValueError(
        "WTDR2D missing count differs from CDC."
    )


print(
    "PASS: WTDR2D matches CDC."
)


# =================================================================================================
# 17. DRDINT CDC CHECK
# =================================================================================================

drdint_counts = (

    day1["DRDINT"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


print(
    "\n--- CDC DRDINT CHECK ---"
)


print(
    drdint_counts
)


if drdint_counts != EXPECTED_DRDINT:

    raise ValueError(
        "DRDINT distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRDINT}\n\n"
        f"Actual:\n{drdint_counts}"
    )


print(
    "PASS: DRDINT matches CDC."
)


# =================================================================================================
# 18. DRABF CDC CHECK
# =================================================================================================

drabf_counts = (

    day1["DRABF"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


print(
    "\n--- CDC DRABF CHECK ---"
)


print(
    drabf_counts
)


if drabf_counts != EXPECTED_DRABF:

    raise ValueError(
        "DRABF distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRABF}\n\n"
        f"Actual:\n{drabf_counts}"
    )


print(
    "PASS: DRABF matches CDC."
)


# =================================================================================================
# 19. SAVE ORIGINAL COLUMN ORDER
# =================================================================================================

original_day1_columns = (
    day1.columns.tolist()
)


original_day2_columns = (
    day2.columns.tolist()
)


# =================================================================================================
# 20. REMOVE ONLY REDUNDANT DAY 2 SHARED COPIES
# =================================================================================================

day2_for_merge = (

    day2.drop(
        columns=SHARED_COLUMNS
    )

)


print(
    "\nRemoved redundant DR2TOT_H copies:"
)


for col in SHARED_COLUMNS:

    print(
        " -",
        col
    )


# =================================================================================================
# 21. VERIFY NO OTHER COLUMN COLLISIONS
# =================================================================================================

remaining_overlap = sorted(

    (
        set(
            day1.columns
        )

        &

        set(
            day2_for_merge.columns
        )
    )

    -

    {
        "SEQN"
    }

)


if remaining_overlap:

    raise ValueError(
        "Unexpected overlapping columns remain before merge:\n"
        + str(
            remaining_overlap
        )
    )


print(
    "PASS: No unexpected variable collisions remain."
)


# =================================================================================================
# 22. MERGE DAY 1 + DAY 2
# =================================================================================================

dietary = (

    day1.merge(

        day2_for_merge,

        on="SEQN",

        how="outer",

        validate="one_to_one"

    )

)


#
# Convert merge-generated NaN back to blank strings.
#

dietary = (
    dietary.fillna("")
)


print(
    "\nPASS: DR1TOT_H + DR2TOT_H merge completed."
)


# =================================================================================================
# 23. FINAL PARTICIPANT CHECK
# =================================================================================================

if len(dietary) != EXPECTED_ROWS:

    raise ValueError(
        "Merged Dietary row count changed.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(dietary):,}"
    )


if (
    dietary["SEQN"]
    .nunique()
    !=
    EXPECTED_ROWS
):

    raise ValueError(
        "Merged unique SEQN count changed."
    )


if (
    dietary["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Merged Dietary contains duplicate SEQN."
    )


if (
    dietary["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Merged Dietary contains blank SEQN."
    )


if (
    dietary["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Merged Dietary contains SEQN '.0'."
    )


print(
    "PASS: Final Dietary file contains "
    "9,813 unique participants."
)


# =================================================================================================
# 24. DUPLICATE COLUMN NAME CHECK
# =================================================================================================

duplicate_column_names = (

    dietary.columns[
        dietary.columns.duplicated()
    ]

    .tolist()

)


if duplicate_column_names:

    raise ValueError(
        "Duplicate column names found:\n"
        + str(
            duplicate_column_names
        )
    )


print(
    "PASS: No duplicate column names."
)


# =================================================================================================
# 25. ENGINEERED DUPLICATE CHECK
# =================================================================================================

BAD_DUPLICATE_COLUMNS = [

    "WTDRD1_DAY_2_TOTAL_INTAKE",

    "WTDR2D_DAY_2_TOTAL_INTAKE",

    "DRABF_DAY_2_TOTAL_INTAKE",

    "DRDINT_DAY_2_TOTAL_INTAKE",

    "WTDRD1_X",

    "WTDRD1_Y",

    "WTDR2D_X",

    "WTDR2D_Y",

    "DRABF_X",

    "DRABF_Y",

    "DRDINT_X",

    "DRDINT_Y"

]


existing_bad_columns = [

    col

    for col in BAD_DUPLICATE_COLUMNS

    if col in dietary.columns

]


if existing_bad_columns:

    raise ValueError(
        "Engineered Dietary duplicate columns remain:\n"
        + str(
            existing_bad_columns
        )
    )


print(
    "PASS: No engineered shared-variable duplicates."
)


# =================================================================================================
# 26. FINAL LOGICAL COLUMN ORDER
# =================================================================================================

SHARED_ORDER = [

    "SEQN",

    "WTDRD1",

    "WTDR2D",

    "DRABF",

    "DRDINT"

]


DAY1_UNIQUE_COLUMNS = [

    col

    for col in original_day1_columns

    if col not in SHARED_ORDER

]


DAY2_UNIQUE_COLUMNS = [

    col

    for col in original_day2_columns

    if (
        col != "SEQN"
        and
        col not in SHARED_COLUMNS
    )

]


FINAL_COLUMN_ORDER = (

    SHARED_ORDER

    +

    DAY1_UNIQUE_COLUMNS

    +

    DAY2_UNIQUE_COLUMNS

)


if (
    len(FINAL_COLUMN_ORDER)
    !=
    len(set(FINAL_COLUMN_ORDER))
):

    raise ValueError(
        "FINAL_COLUMN_ORDER contains duplicate fields."
    )


if (
    set(FINAL_COLUMN_ORDER)
    !=
    set(dietary.columns)
):

    raise ValueError(
        "Final column-order structure does not match dataframe."
    )


dietary = (
    dietary[
        FINAL_COLUMN_ORDER
    ]
)


print(
    "PASS: Final column order created."
)


# =================================================================================================
# 27. DISPLAY FINAL STRUCTURE
# =================================================================================================

print(
    "\n--- FINAL STRUCTURE ---"
)


print(
    "Shared fields:",
    len(SHARED_ORDER)
)


print(
    "Day 1 unique fields:",
    len(DAY1_UNIQUE_COLUMNS)
)


print(
    "Day 2 unique fields:",
    len(DAY2_UNIQUE_COLUMNS)
)


print(
    "Total fields:",
    len(dietary.columns)
)


# =================================================================================================
# 28. DAY 1 EXACT SOURCE-TEXT FIDELITY
# =================================================================================================

print(
    "\n--- DR1TOT_H SOURCE FIDELITY ---"
)


day1_check = (

    day1.merge(

        dietary[
            original_day1_columns
        ],

        on="SEQN",

        how="left",

        suffixes=(
            "_SOURCE",
            "_MERGED"
        ),

        validate="one_to_one"

    )

)


day1_failures = []


for col in original_day1_columns:

    if col == "SEQN":

        continue


    source_values = (
        day1_check[
            f"{col}_SOURCE"
        ]
        .fillna("")
    )


    merged_values = (
        day1_check[
            f"{col}_MERGED"
        ]
        .fillna("")
    )


    mismatch_count = int(

        (
            source_values
            !=
            merged_values
        )

        .sum()

    )


    if mismatch_count > 0:

        day1_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day1_failures:

    raise ValueError(
        "DR1TOT_H values changed during merge:\n"
        + str(
            day1_failures
        )
    )


print(
    "PASS: Every DR1TOT_H value preserved text-for-text."
)


# =================================================================================================
# 29. DAY 2 UNIQUE FIELD TEXT FIDELITY
# =================================================================================================

print(
    "\n--- DR2TOT_H SOURCE FIDELITY ---"
)


day2_check_columns = (

    ["SEQN"]

    +

    DAY2_UNIQUE_COLUMNS

)


day2_check = (

    day2[
        day2_check_columns
    ]

    .merge(

        dietary[
            day2_check_columns
        ],

        on="SEQN",

        how="left",

        suffixes=(
            "_SOURCE",
            "_MERGED"
        ),

        validate="one_to_one"

    )

)


day2_failures = []


for col in DAY2_UNIQUE_COLUMNS:

    source_values = (
        day2_check[
            f"{col}_SOURCE"
        ]
        .fillna("")
    )


    merged_values = (
        day2_check[
            f"{col}_MERGED"
        ]
        .fillna("")
    )


    mismatch_count = int(

        (
            source_values
            !=
            merged_values
        )

        .sum()

    )


    if mismatch_count > 0:

        day2_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day2_failures:

    raise ValueError(
        "DR2TOT_H values changed during merge:\n"
        + str(
            day2_failures
        )
    )


print(
    "PASS: Every unique DR2TOT_H value "
    "preserved text-for-text."
)


# =================================================================================================
# 30. DIETARY DECIMAL PRESERVATION
# =================================================================================================

#
# Diagnostic only.
#
# No rounding or numeric conversion occurs.
#

REPRESENTATIVE_DECIMAL_FIELDS = [

    "WTDRD1",

    "WTDR2D",

    "DR1TPROT",

    "DR1TCARB",

    "DR1TP204",

    "DR1_320Z",

    "DR2TPROT",

    "DR2TCARB",

    "DR2TP204",

    "DR2_320Z"

]


print(
    "\n--- DIETARY DECIMAL PRESERVATION ---"
)


for col in REPRESENTATIVE_DECIMAL_FIELDS:

    if col not in dietary.columns:

        continue


    populated_count = int(
        dietary[col]
        .ne("")
        .sum()
    )


    decimal_token_count = int(

        (
            dietary[col].ne("")
            &
            dietary[col]
            .str.contains(
                ".",
                regex=False
            )
        )

        .sum()

    )


    print(
        f"{col}: "
        f"populated={populated_count:,}, "
        f"decimal tokens={decimal_token_count:,}"
    )


print(
    "PASS: Dietary values were not rounded "
    "or integerized during merge."
)


# =================================================================================================
# 31. KNOWN XPORT TINY-ARTIFACT CHECK
# =================================================================================================

TINY_TOKEN = (
    "5.397605346934028e-79"
)


tiny_found = {}


for col in dietary.columns:

    if col == "SEQN":

        continue


    count = int(

        dietary[col]
        .str.lower()
        .eq(
            TINY_TOKEN.lower()
        )
        .sum()

    )


    if count > 0:

        tiny_found[
            col
        ] = count


if tiny_found:

    raise ValueError(
        "Known XPORT tiny-zero artifacts remain:\n"
        + str(
            tiny_found
        )
    )


print(
    "PASS: No known XPORT tiny-zero artifacts."
)


# =================================================================================================
# 32. SAVE FINAL CSV
# =================================================================================================

#
# CRITICAL:
#
# No float_format=
#
# No rounding.
#
# No numeric conversion.
#

dietary.to_csv(

    OUTPUT_PATH,

    index=False

)


print(
    "\nSaved final Dietary file:"
)


print(
    OUTPUT_PATH
)


# =================================================================================================
# 33. RE-READ FINAL CSV AS TEXT
# =================================================================================================

final_df = pd.read_csv(

    OUTPUT_PATH,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 34. POST-SAVE STRUCTURE CHECK
# =================================================================================================

if final_df.shape != dietary.shape:

    raise ValueError(
        "Saved CSV shape changed.\n"
        f"Before: {dietary.shape}\n"
        f"After:  {final_df.shape}"
    )


if (
    final_df.columns.tolist()
    !=
    dietary.columns.tolist()
):

    raise ValueError(
        "Saved CSV column order changed."
    )


print(
    "PASS: Saved CSV structure preserved."
)


# =================================================================================================
# 35. EXACT PRE-SAVE / POST-SAVE TEXT FIDELITY
# =================================================================================================

post_save_failures = []


for col in dietary.columns:

    mismatch_count = int(

        (
            dietary[col]
            !=
            final_df[col]
        )

        .sum()

    )


    if mismatch_count > 0:

        post_save_failures.append(
            (
                col,
                mismatch_count
            )
        )


if post_save_failures:

    raise ValueError(
        "CSV text changed during save:\n"
        + str(
            post_save_failures
        )
    )


print(
    "PASS: Exact pre-save/post-save text fidelity."
)


# =================================================================================================
# 36. FINAL SEQN CHECK
# =================================================================================================

if (
    final_df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    final_df["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if (
    final_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV contains SEQN '.0'."
    )


if (

    ~final_df["SEQN"]
    .str.fullmatch(r"\d+")

).any():

    raise ValueError(
        "Final CSV contains invalid SEQN."
    )


print(
    "PASS: Final SEQN integrity."
)


# =================================================================================================
# 37. FINAL ENGINEERED-DUPLICATE CHECK
# =================================================================================================

final_bad_columns = [

    col

    for col in BAD_DUPLICATE_COLUMNS

    if col in final_df.columns

]


if final_bad_columns:

    raise ValueError(
        "Engineered duplicate fields remain:\n"
        + str(
            final_bad_columns
        )
    )


print(
    "PASS: No redundant Day 2 shared-field copies."
)


# =================================================================================================
# 38. FINAL CDC WTDRD1 CHECK
# =================================================================================================

final_wtdrd1 = pd.to_numeric(

    final_df["WTDRD1"]
    .where(
        final_df["WTDRD1"] != "",
        np.nan
    ),

    errors="raise"

)


assert int(
    (
        final_wtdrd1 > 0
    )
    .sum()
) == 8661


assert int(
    final_wtdrd1
    .eq(0)
    .sum()
) == 1152


assert int(
    final_wtdrd1
    .isna()
    .sum()
) == 0


print(
    "PASS: Final WTDRD1:"
)


print(
    "  Positive = 8,661"
)


print(
    "  Zero     = 1,152"
)


print(
    "  Missing  = 0"
)


# =================================================================================================
# 39. FINAL CDC WTDR2D CHECK
# =================================================================================================

final_wtdr2d = pd.to_numeric(

    final_df["WTDR2D"]
    .where(
        final_df["WTDR2D"] != "",
        np.nan
    ),

    errors="raise"

)


assert int(
    (
        final_wtdr2d > 0
    )
    .sum()
) == 7574


assert int(
    final_wtdr2d
    .eq(0)
    .sum()
) == 1087


assert int(
    final_wtdr2d
    .isna()
    .sum()
) == 1152


print(
    "PASS: Final WTDR2D:"
)


print(
    "  Positive = 7,574"
)


print(
    "  Zero     = 1,087"
)


print(
    "  Missing  = 1,152"
)


# =================================================================================================
# 40. FINAL DRDINT CHECK
# =================================================================================================

final_drdint = (

    final_df["DRDINT"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


if final_drdint != EXPECTED_DRDINT:

    raise ValueError(
        "Final DRDINT differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRDINT}\n\n"
        f"Actual:\n{final_drdint}"
    )


print(
    "PASS: Final DRDINT matches CDC."
)


# =================================================================================================
# 41. FINAL DRABF CHECK
# =================================================================================================

final_drabf = (

    final_df["DRABF"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


if final_drabf != EXPECTED_DRABF:

    raise ValueError(
        "Final DRABF differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRABF}\n\n"
        f"Actual:\n{final_drabf}"
    )


print(
    "PASS: Final DRABF matches CDC."
)


# =================================================================================================
# 42. FINAL TINY-ARTIFACT CHECK
# =================================================================================================

final_tiny_found = {}


for col in final_df.columns:

    if col == "SEQN":

        continue


    count = int(

        final_df[col]
        .str.lower()
        .eq(
            TINY_TOKEN.lower()
        )
        .sum()

    )


    if count > 0:

        final_tiny_found[
            col
        ] = count


if final_tiny_found:

    raise ValueError(
        "Tiny XPORT artifacts remain after save:\n"
        + str(
            final_tiny_found
        )
    )


print(
    "PASS: No tiny XPORT artifacts after save."
)


# =================================================================================================
# 43. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2013-2014 DIETARY MERGE COMPLETE"
)


print(
    "=" * 110
)


print(
    "Cycle:"
)


print(
    "  2013-2014"
)


print(
    ""
)


print(
    "Source files:"
)


print(
    "  DR1TOT_H_corrected.csv"
)


print(
    "  DR2TOT_H_corrected.csv"
)


print(
    ""
)


print(
    "Rows:",
    f"{len(final_df):,}"
)


print(
    "Unique SEQN:",
    f"{final_df['SEQN'].nunique():,}"
)


print(
    "Columns:",
    f"{len(final_df.columns):,}"
)


print(
    ""
)


print(
    "CDC Day 1:"
)


print(
    "  Positive WTDRD1 = 8,661"
)


print(
    "  WTDRD1 zero     = 1,152"
)


print(
    "  WTDRD1 missing  = 0"
)


print(
    ""
)


print(
    "CDC Day 2:"
)


print(
    "  Positive WTDR2D = 7,574"
)


print(
    "  WTDR2D zero     = 1,087"
)


print(
    "  WTDR2D missing  = 1,152"
)


print(
    ""
)


print(
    "DRDINT:"
)


print(
    "  Day 1 only       = 1,087"
)


print(
    "  Day 1 + Day 2    = 7,574"
)


print(
    "  Missing          = 1,152"
)


print(
    ""
)


print(
    "DRABF:"
)


print(
    "  Yes              =   134"
)


print(
    "  No               = 8,527"
)


print(
    "  Missing          = 1,152"
)


print(
    ""
)


print(
    "Shared fields retained once:"
)


print(
    "  WTDRD1"
)


print(
    "  WTDR2D"
)


print(
    "  DRABF"
)


print(
    "  DRDINT"
)


print(
    ""
)


print(
    "Duplicate SEQN: 0"
)


print(
    "SEQN '.0': 0"
)


print(
    "Engineered Dietary duplicates: 0"
)


print(
    "Known XPORT tiny artifacts: 0"
)


print(
    "Day 1 retained-value changes: 0"
)


print(
    "Day 2 retained-value changes: 0"
)


print(
    "Dietary decimal reformatting during merge: 0"
)


print(
    "Pre-save/post-save text fidelity: PASS"
)


print(
    "=" * 110
)

NHANES 2013-2014 DIETARY DAY 1 + DAY 2 MERGE
DR1TOT_H shape: (9813, 168)
DR2TOT_H shape: (9813, 85)
PASS: SEQN exists in both files.
PASS: Both DR1TOT_H and DR2TOT_H contain 9,813 rows.

--- SEQN CHECK: DR1TOT_H ---
Rows: 9,813
Unique SEQN: 9,813
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN: 0
PASS

--- SEQN CHECK: DR2TOT_H ---
Rows: 9,813
Unique SEQN: 9,813
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN: 0
PASS

PASS: SEQN integrity validated in both files.

--- PARTICIPANT SET CHECK ---
Only in DR1TOT_H: 0
Only in DR2TOT_H: 0
PASS: Both files contain the same 9,813 participants.

--- SHARED VARIABLES DETECTED ---
['WTDRD1', 'WTDR2D', 'DRABF', 'DRDINT']
PASS: Exactly four expected shared variables found.

--- SHARED VARIABLE FIDELITY CHECK ---

WTDRD1
  Both populated: 9,813
  Missing-position mismatches: 0
  True numeric mismatches: 0
  Text-format differences: 0

WTDR2D
  Both populated: 8,661
  Missing-position mismatches: 0
  True numeric mism